# csvの読み込み

In [0]:
SELECT *
FROM csv.`/Volumes/users/yukiteru_koide/learning/sample.csv`

# csvからテーブルの作成

In [0]:
USE CATALOG users;
USE SCHEMA yukiteru_koide;

CREATE OR REPLACE TABLE sample AS
SELECT *
FROM read_files(
  '/Volumes/users/yukiteru_koide/learning/sample.csv',
  format => 'csv',
  header => true,
  schema => '
    Name STRING,
    Age INT,
    _rescued_data TIMESTAMP'
)


In [0]:
SELECT *
FROM sample

# view

## 標準 View（永続・スキーマ内）

In [0]:
USE CATALOG users;
USE SCHEMA yukiteru_koide;

-- 再実行しても安全
DROP VIEW IF EXISTS v_customer_all;

CREATE VIEW v_customer_all AS
SELECT *
FROM users.yukiteru_koide.customer;


## Temporary View（セッション限定・スキーマ外）


In [0]:
-- TEMP VIEW はセッションが切れると消えます
-- カタログ/スキーマ指定は不要（オブジェクトはワークスペースのセッション内に存在）
CREATE OR REPLACE TEMP VIEW tmp_customer AS
SELECT *
FROM users.yukiteru_koide.customer;

-- 使い方
SELECT * FROM tmp_customer;


## Global Temporary View（ワークスペース内グローバル・セッション共有）

In [0]:
-- GLOBAL TEMP VIEW はスキーマ名が 'global_temp'
CREATE OR REPLACE GLOBAL TEMP VIEW gtmp_customer AS
SELECT *
FROM users.yukiteru_koide.customer;

-- 使い方（必ず global_temp をプレフィックスに）
SELECT * FROM global_temp.gtmp_customer;


## マテリアライズド・ビュー（結果を保存・自動/手動リフレッシュ）

In [0]:
USE CATALOG users;
USE SCHEMA yukiteru_koide;

-- 再実行しても安全
DROP MATERIALIZED VIEW IF EXISTS mv_customer_basic;

-- よく参照する列だけに絞る（例）
CREATE MATERIALIZED VIEW mv_customer_basic AS
SELECT customer_id, name, signup_date
FROM users.yukiteru_koide.customer;

-- 手動リフレッシュ（必要に応じて）
-- REFRESH MATERIALIZED VIEW mv_customer_basic;

-- 利用
SELECT * FROM mv_customer_basic;


In [0]:
USE CATALOG users;
USE SCHEMA yukiteru_koide;

CREATE OR REPLACE MATERIALIZED VIEW mv_customer_hourly
SCHEDULE EVERY 1 HOUR
AS
SELECT
  customer_id,
  name,
  signup_date
FROM users.yukiteru_koide.customer;


## Dynamic View（動的マスキング/行フィルタ）
- 例1: PII カラムをグループでマスク制御
- 例2: グループに応じた行レベル制御（リージョン）

In [0]:
USE CATALOG users;
USE SCHEMA yukiteru_koide;

DROP VIEW IF EXISTS v_customer_secure;

CREATE VIEW v_customer_secure AS
SELECT
  customer_id,
  name,
  -- ①メールの動的マスキング例：
  -- 'pii_readers' アカウントグループのメンバーだけ生値を表示
  CASE
    WHEN is_account_group_member('pii_readers') THEN email
    ELSE '*** MASKED ***'
  END AS email,
  -- ②電話番号の動的マスキング例（ハッシュ化などでも可）
  CASE
    WHEN is_account_group_member('pii_readers') THEN phone
    ELSE NULL
  END AS phone,
  region,
  signup_date
FROM users.yukiteru_koide.customer
-- ③行レベル制御（APAC 閲覧グループのみ APAC 行を見せる例）
WHERE
  -- APAC 専用グループのメンバーは APAC だけ閲覧
  (is_account_group_member('apac_readers') AND region = 'APAC')
  -- グローバル閲覧者は全件閲覧
  OR is_account_group_member('global_readers');
